# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
removed = df.duplicated().sum()
clean = df.drop_duplicates().copy()
old_price = df['price'].astype(str)
formatted_price = (old_price
                   .str.replace('$', '', regex=False)
                   .str.replace(',', '', regex=False)
                   .str.strip())
df['price'] = formatted_price.astype(float)
old_revenue = (df['price'] * df['qty']).sum()
df
log('duplicates', 'dropped duplicates', removed)

[duplicates] dropped duplicates (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
old_price = clean['price'].astype(str)

formatted_price = (old_price
                   .str.replace('$', '', regex=False)
                   .str.replace(',', '', regex=False)
                   .str.strip())

rows_affected = len(clean)

clean['price'] = formatted_price.astype(float)

one_revenue = (clean['price'] * clean['qty']).sum()

log('price', 'reformatted price into floats', rows_affected)
clean

[price] reformatted price into floats (300 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,7.5
3,3,Cheeseburger,Food,NaN,7.5
4,4,cheese burger,Apparel,1.0,7.5
...,...,...,...,...,...
295,295,rain poncho,food,3.0,6.0
296,296,cheese burger,RainGear,-1.0,6.0
297,297,Foam Finger,food,1.0,7.5
298,298,Rain Poncho,Merch,1.0,7.5


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()
zero_negative = (clean['qty'] <= 0).sum()

clean = clean.dropna(subset=['qty'])
clean = clean[clean['qty'] > 0]

two_revenue = (clean['price'] * clean['qty']).sum()

log('qty', 'dropped rows with missing and negative/zero quantity', missing + zero_negative)

clean

[qty] dropped rows with missing and negative/zero quantity (25 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,7.5
4,4,cheese burger,Apparel,1.0,7.5
5,5,Foam Finger,food,3.0,6.0
...,...,...,...,...,...
294,294,rain poncho,Merch,2.0,6.0
295,295,rain poncho,food,3.0,6.0
297,297,Foam Finger,food,1.0,7.5
298,298,Rain Poncho,Merch,1.0,7.5


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(clean['item'].value_counts())
original_items = clean['item'].copy()
clean['item'] = (clean['item'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True))
print(clean['item'].value_counts())
ITEM_MAP = {
    "cheese burger": "cheeseburger"
}
clean['item'] = clean['item'].replace(ITEM_MAP)
changed = (original_items != clean['item']).sum()
log('item', 'normalize and deduplicated items', changed)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
item
foam finger      97
rain poncho      91
cheese burger    44
cheeseburger     43
Name: count, dtype: int64
[item] normalize and deduplicated items (193 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(clean['category'].value_counts())
original_categories = clean['category'].copy()
clean['category'] = (clean['category'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True))
print(clean['category'].value_counts())
CATEGORY_MAP = {
    'apparel': 'merch'
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)
print(clean['category'].value_counts())
changed = (original_categories != clean['category']).sum()
log('category', 'normalized and deduplicated categories', changed)

category
Merch        51
Food         51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
category
food        95
raingear    86
merch       51
apparel     43
Name: count, dtype: int64
category
food        95
merch       94
raingear    86
Name: count, dtype: int64
[category] normalized and deduplicated categories (231 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
# TODO: assert something about item
assert set(clean['item'].unique()) == {'foam finger','rain poncho','cheeseburger'}

assert set(clean['category'].unique()) == {'food','merch','raingear'}
# TODO: assert something about category
print('clean:', clean.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
clean['revenue'] = clean['price'] * clean['qty']
print(f"Overall total revenue: {clean['revenue'].sum():.2f}")
print(clean['revenue'].sum(), one_revenue, two_revenue, old_revenue)
by_category = pd.DataFrame(clean.groupby('category')['revenue'].sum().round(2))
by_category = by_category.sort_values('revenue', ascending=False)
by_category

Overall total revenue: 4740.00
4740.0 4594.5 4740.0 4852.5


,revenue
category,
food,1656.0
merch,1572.0
raingear,1512.0


**What I would tell the vendor:** I would stock more of food since that had the highest revenue out of all the categories

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped duplicates,15
1,price,reformatted price into floats,300
2,qty,dropped rows with missing and negative/zero qu...,25
3,item,normalize and deduplicated items,193
4,category,normalized and deduplicated categories,231


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.



```
# This is formatted as code
```

a) The cleaning step that changed my revenue total the most was dropping duplicated rows. With the original data frame, the revenue would have been 4852.5. However, deduplicating those values resulted in a total revenue of 4594.5 which is a 258 reduction. While the final number ended up being 4740, that initial 258 dollar reduction was the biggest change.

b) I chose to drop the negative values - the other choice would have been to keep it to represent refunds. This would have decreased my reported revenue from 4740 -> 4594.50 dollars. I made the choice to drop negative values since I felt it would be more beneficial to keep the number as a gross revenue rather than go straight for the net revenue including refunds as it would show analysts the difference between our gross and net, and likely drive investigation towards where customers might have refunded.